# The Climate is Changing: How is the Research?

## Exploring Trends in Climate Articles: 2013 to 2023

#### This notebook explores BOW and TFIDF concepts within the corpus.

#### Written by Rafael Alvarado(1) and Caroline Kranefuss(1).

(1) University of Virginia, 2026

### Imports

In [36]:
# General imports
import pandas as pd 
import numpy as np 
from numpy.linalg import norm
import os
import sys

# Plotting
import plotly.express as px

# Project-specific imports
import glob
from lxml import etree
from glob import glob
import re
import nltk
nltk_resources = [
    'tokenizers/punkt', 
    'averaged_perceptron_tagger_eng',
    'corpora/stopwords', 
    'help/tagsets'
]

for resource in nltk_resources:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(resource)
        
        
# ----File Stitching----
# If not in repo home folder, cd back 
if os.path.basename(os.getcwd()) != "evolving_sentiment_climate":
    os.chdir('..')
# If a file is in /sources/, access it by telling the system to look at that path as well as current path
sys.path.append(os.path.join(os.getcwd(), 'sources'))
source_dir = "sources"
source_files_paths = glob(f"{source_dir}/*.xml")
# Same for csvs
sys.path.append(os.path.join(os.getcwd(), 'csvs'))
csvs_dir = "csvs"
csvs_files_paths = glob(f"{csvs_dir}/*.csv")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\student\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [37]:
DOCS = pd.read_csv("csvs/DOC/DOC.csv")

In [38]:
%%capture
%run notebooks/LDA.ipynb

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\student\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [39]:
%%capture
%run notebooks/PCA.ipynb

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\student\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [40]:
# Join with lib to get years into index, reformat
THETA_years = THETA.join(LIB, on=THETA.index.get_level_values('doc_id'))
THETA_years.reset_index(inplace=True)
THETA_years.set_index(['year', 'doc_id'], inplace=True)
THETA_years.drop(columns=['doc_title', 'doc_1st_author', 'sent_str', 'num_words', 'num_chars'], inplace=True)
THETA_years

key_0       T00       T01       T02       T03       T04  \
year doc_id                                                            
2018 0           0  0.011095  0.000035  0.000035  0.000035  0.000035   
2023 1           1  0.000017  0.000017  0.963597  0.009445  0.000017   
2018 2           2  0.000033  0.000033  0.650169  0.050454  0.000033   
     3           3  0.000029  0.000029  0.000029  0.000029  0.000029   
2013 4           4  0.000058  0.000058  0.000058  0.000058  0.000058   
...            ...       ...       ...       ...       ...       ...   
2023 87         87  0.000020  0.000020  0.000020  0.000020  0.109333   
     88         88  0.000019  0.981585  0.000019  0.000019  0.000019   
2018 89         89  0.000019  0.000019  0.000019  0.000019  0.000019   
2023 90         90  0.000024  0.038390  0.000024  0.000024  0.000024   
     91         91  0.034861  0.000032  0.073059  0.000032  0.000032   

                  T05       T06       T07       T08  ...       T10       T11  \
year doc_id                                          ...                       
2018 0       0.000035  0.000035  0.011427  0.000035  ...  0.109965  0.000035   
2023 1       0.000017  0.020529  0.000017  0.000017  ...  0.000017  0.000017   
2018 2       0.000033  0.048960  0.000033  0.000033  ...  0.089562  0.000033   
     3       0.000029  0.000029  0.000029  0.000029  ...  0.000029  0.000029   
2013 4       0.000058  0.098319  0.041469  0.000058  ...  0.000058  0.000058   
...               ...       ...       ...       ...  ...       ...       ...   
2023 87      0.000020  0.000020  0.000020  0.000020  ...  0.733123  0.000020   
     88      0.000019  0.000019  0.000019  0.000019  ...  0.000019  0.000019   
2018 89      0.000019  0.000019  0.000019  0.000019  ...  0.000019  0.000019   
2023 90      0.000024  0.000024  0.000024  0.000024  ...  0.000024  0.770142   
     91      0.000032  0.645685  0.000032  0.027656  ...  0.000032  0.000032   

                  T12       T13       T14       T15       T16       T17  \
year doc_id                                                               
2018 0       0.000035  0.032310  0.000035  0.088021  0.037922  0.000035   
2023 1       0.000017  0.000017  0.000017  0.006162  0.000017  0.000017   
2018 2       0.032452  0.000033  0.000033  0.000033  0.044731  0.061068   
     3       0.000029  0.000029  0.000029  0.000029  0.000029  0.999442   
2013 4       0.534401  0.191730  0.000058  0.000058  0.012801  0.000058   
...               ...       ...       ...       ...       ...       ...   
2023 87      0.000020  0.000020  0.028116  0.000020  0.000020  0.000020   
     88      0.018068  0.000019  0.000019  0.000019  0.000019  0.000019   
2018 89      0.000019  0.000019  0.999634  0.000019  0.000019  0.000019   
2023 90      0.000024  0.000024  0.006188  0.000024  0.112353  0.000024   
     91      0.040578  0.008181  0.000032  0.117883  0.000032  0.006374   

                  T18       T19  
year doc_id                      
2018 0       0.000035  0.708800  
2023 1       0.000017  0.000017  
2018 2       0.000033  0.000033  
     3       0.000029  0.000029  
2013 4       0.000058  0.000058  
...               ...       ...  
2023 87      0.000020  0.000020  
     88      0.000019  0.000019  
2018 89      0.000019  0.000019  
2023 90      0.000024  0.000024  
     91      0.000032  0.045369  

[92 rows x 21 columns]

In [41]:
# Define engine
pca_engine = PCA(n_components=3)
# Run engine on the TRANSPOSE of THETA
engine_run = pca_engine.fit_transform(THETA.T)
# Define index
index_TCM = THETA.T.index
# Turn into df
TCM = pd.DataFrame(engine_run, index=index_TCM)
TCM.rename(columns={0:'PC0', 1:'PC1', 2:'PC2'}, inplace=True)
# View
TCM

,PC0,PC1,PC2
topic_id,,,
T00,-0.158325,-0.167703,-0.103067
T01,-0.202740,-0.077344,-0.128274
T02,-0.240039,0.060570,-0.067228
T03,0.023317,0.131697,-0.356225
T04,-0.255934,-0.048157,-0.347527
T05,-0.100084,0.021296,-0.236513
T06,-0.692099,0.510531,0.950571
T07,0.296300,-0.156357,0.036122
T08,-0.162609,-0.088335,-0.125605


In [42]:
# Make a components table that essentially documents the explained variance of each term
# I.e., how much each term is part of any given component
# This will help in defining the components' meanings 
COMPS = pd.DataFrame(pca_engine.components_.T * np.sqrt(pca_engine.explained_variance_))
# Formatting and settings (columns, index, transpose for easy viewing)
COMPS.columns = ["PC{}".format(i) for i in COMPS.columns]
COMPS.index = THETA.T.columns

COMPS

,PC0,PC1,PC2
doc_id,,,
0,0.079187,-0.079836,0.059092
1,-0.027230,0.007790,-0.005471
2,-0.007237,0.020029,-0.000894
3,-0.024762,-0.007244,-0.021330
4,-0.011046,0.011396,-0.019168
...,...,...,...
87,0.124296,0.047643,-0.010682
88,-0.022316,-0.009120,-0.016501
89,-0.021402,0.001945,-0.042623


In [54]:
# Adapting plotting functions
# A function to visualize the principal components
# Inputs: The DCM df, PCs a and b
def vis_pcs(DCM, a, b):
    return px.scatter(DCM, 
                      f"PC{a}", f"PC{b}", 
                      hover_name=DCM.index.get_level_values('topic_id').astype(str),
                      marginal_x='box', height=800)

# Another function for visualization
# Inputs: The COMPS df, PCs and b
def vis_loadings(COMPS, a, b):
    return px.scatter(COMPS, f"PC{a}", f"PC{b}", 
                      text=COMPS.index,
                      hover_name=COMPS.index,
                      marginal_x='box', 
                      height=800)

In [55]:
vis_pcs(TCM, 0, 1)

In [ ]:
vis_loadings(COMPS, 0,1)